# Cryptographic Techniques in the Secure RPM Platform
### Interactive Verification & Execution Notebook

This notebook imports the project's cryptographic functions directly from the `backend/app` codebase and demonstrates:
1. **Field-Level Data-at-Rest Encryption (AES-256-GCM)**: How patient vitals arrive, how they are encrypted in the DB, and how authorized decryption works.
2. **Network Transmission & Data Minimization**: What the incoming HTTP REST packet looks like vs. the real-time Server-Sent Events (SSE) push stream.
3. **User Authentication (Argon2id)**: Memory-hard password hashing and verification.
4. **Session Token Security (SHA-256 Preimage Resistance)**: Why database leaks cannot hijack user sessions.
5. **Tamper-Evident Audit Logging (SHA-256 Hash Chaining)**: How blockchain-style log linking detects retroactive tampering.
6. **Quantum Circuit Structural Fingerprinting**: How quantum job execution is audited cryptographically without logging medical data.


In [1]:
import os
import sys
import json
import uuid
import datetime

# Add backend directory to module search path
backend_path = os.path.abspath("backend")
if backend_path not in sys.path:
    sys.path.insert(0, backend_path)

# Set demo environment variables
os.environ.setdefault("DATABASE_URL", "sqlite:///:memory:")
os.environ.setdefault("APP_SECRET_KEY", "demo-secret-key-32-bytes-minimum-length!")
os.environ.setdefault("FIELD_ENCRYPTION_KEY_B64", "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA=")

print("✓ Environment initialized and backend modules loaded.")


✓ Environment initialized and backend modules loaded.


---
## 1. Data at Rest: Field-Level AES-256-GCM Encryption
**File:** `backend/app/core/crypto.py`  
**Purpose:** Ensure sensitive patient measurements are never stored in plaintext inside the database (`dev.db` / PostgreSQL). A database handler or rogue admin inspecting the tables only sees opaque binary ciphertext.


In [2]:
from app.core.crypto import encrypt_value, decrypt_value

# Step 1: Patient submits a vital reading (e.g. Heart Rate)
incoming_patient_reading = "78.5"  # bpm
print(f"[1] Incoming Patient Data from Device/Form: {incoming_patient_reading} bpm (Plaintext)")

# Step 2: Backend encrypts the value before database insertion
encrypted_blob = encrypt_value(incoming_patient_reading)
print(f"\n[2] Stored in Database (measurements.encrypted_value):")
print(f"    Raw bytes: {encrypted_blob[:20]}... (total {len(encrypted_blob)} bytes)")
print(f"    Hex representation seen by DBA: {encrypted_blob.hex()[:50]}...")

# Step 3: Demonstrate Probabilistic Encryption (Random Nonce per call)
# Encrypting the exact same reading a second later produces a completely different ciphertext!
encrypted_blob_2 = encrypt_value(incoming_patient_reading)
print(f"\n[3] Re-encrypting identical value ('78.5'):")
print(f"    Ciphertext 1 hex: {encrypted_blob.hex()[:40]}...")
print(f"    Ciphertext 2 hex: {encrypted_blob_2.hex()[:40]}...")
print(f"    Are ciphertexts identical? {encrypted_blob == encrypted_blob_2}")
print("    --> Prevents frequency analysis by attackers!")

# Step 4: Authorized Decryption at Application Layer
decrypted_value = decrypt_value(encrypted_blob)
print(f"\n[4] Decrypted value for authorized doctor/patient: {decrypted_value} bpm")
assert decrypted_value == incoming_patient_reading
print("    --> Verification successful: Decrypted matches original plaintext exactly.")


[1] Incoming Patient Data from Device/Form: 78.5 bpm (Plaintext)

[2] Stored in Database (measurements.encrypted_value):
    Raw bytes: b'\xa0\xea\t\xb7\x18t\xd8KX7\xf8^\xf1\x7fB\xc0\xca\xe1W3'... (total 32 bytes)
    Hex representation seen by DBA: a0ea09b71874d84b5837f85ef17f42c0cae157331b52fed0ec...

[3] Re-encrypting identical value ('78.5'):
    Ciphertext 1 hex: a0ea09b71874d84b5837f85ef17f42c0cae15733...
    Ciphertext 2 hex: fd9a1b295dfc79c821bb293eb28bce0b87f14e92...
    Are ciphertexts identical? False
    --> Prevents frequency analysis by attackers!

[4] Decrypted value for authorized doctor/patient: 78.5 bpm
    --> Verification successful: Decrypted matches original plaintext exactly.


---
## 2. Data in Transit & Network Packet Comparison
**Files:** `backend/app/ingest/routes.py`, `backend/app/alerts/engine.py`, `docs/architecture/03-threat-model-additions.md`  
**Purpose:** Demonstrate **Architectural Data Minimization**.  
- Medical vitals are transmitted over authenticated HTTPS REST endpoints.  
- Crucially, real-time Server-Sent Events (SSE) push events **never contain raw vital numbers**, preventing health metrics from leaking into edge proxy logs, load balancers, or push notifications.


In [3]:
# Part A: Incoming Client -> Server REST Packet (Protected by TLS/HTTPS)
rest_request_packet = {
    "method": "POST",
    "endpoint": "/api/v1/measurements",
    "headers": {
        "Host": "rpm-platform.example.com",
        "Content-Type": "application/json",
        "Cookie": "rpm_session=uB4Z7q_9a8X... (HttpOnly, Secure)"
    },
    "body": {
        "measurement_type": "heart_rate",
        "value": 145.0,  # Acute tachycardia spike
        "recorded_at": datetime.datetime.now(datetime.timezone.utc).isoformat()
    }
}
print("=== [A] REST HTTP Ingest Packet (Client -> Backend) ===")
print(json.dumps(rest_request_packet, indent=2))

# Part B: Server -> Browser Realtime Push Packet (SSE Stream)
# Notice: Raw health values (145.0 bpm) are stripped! Only metadata is pushed.
alert_id = str(uuid.uuid4())
patient_id = str(uuid.uuid4())
sse_event_packet = (
    "event: new_alert\n"
    f"data: {json.dumps({'alert_id': alert_id, 'patient_id': patient_id, 'severity': 'high', 'created_at': datetime.datetime.now(datetime.timezone.utc).isoformat()})}\n\n"
)
print("\n=== [B] Realtime SSE Push Packet (Backend -> Doctor Browser) ===")
print(sse_event_packet)
print("Key Observation:")
print("The real-time push payload contains strictly non-clinical metadata (ID, severity, timestamp).")
print("Even if reverse proxies log SSE traffic, zero patient vital statistics are exposed.")


=== [A] REST HTTP Ingest Packet (Client -> Backend) ===
{
  "method": "POST",
  "endpoint": "/api/v1/measurements",
  "headers": {
    "Host": "rpm-platform.example.com",
    "Content-Type": "application/json",
    "Cookie": "rpm_session=uB4Z7q_9a8X... (HttpOnly, Secure)"
  },
  "body": {
    "measurement_type": "heart_rate",
    "value": 145.0,
    "recorded_at": "2026-11-16T16:12:00.910420+00:00"
  }
}

=== [B] Realtime SSE Push Packet (Backend -> Doctor Browser) ===
event: new_alert
data: {"alert_id": "249fd164-facb-4e9e-84de-36e5e1de01b3", "patient_id": "53a32c5d-ad2e-4c89-9463-832557cfb23b", "severity": "high", "created_at": "2026-11-16T16:12:00.910653+00:00"}


Key Observation:
The real-time push payload contains strictly non-clinical metadata (ID, severity, timestamp).
Even if reverse proxies log SSE traffic, zero patient vital statistics are exposed.


---
## 3. User Authentication: Memory-Hard Password Hashing (Argon2id)
**File:** `backend/app/core/security.py`  
**Purpose:** Use salted, memory-hard Argon2id (RFC 9106) hashing so that offline dictionary or GPU/ASIC rainbow-table attacks are computationally infeasible.


In [4]:
from app.core.security import hash_password, verify_password

patient_password = "DemoPatient123!"

# Hash password
hashed = hash_password(patient_password)
print(f"Plaintext Password: {patient_password}")
print(f"Argon2id Hash (Stored in DB):\n  {hashed}")

# Verify correct password
is_valid = verify_password("DemoPatient123!", hashed)
print(f"\nVerification with correct password: {is_valid}")

# Verify incorrect password
is_invalid = verify_password("WrongPassword123!", hashed)
print(f"Verification with wrong password:   {is_invalid}")
assert is_valid is True and is_invalid is False
print("--> Password authentication verified.")


Plaintext Password: DemoPatient123!
Argon2id Hash (Stored in DB):
  $argon2id$v=19$m=65536,t=3,p=4$+PEbP2ckEM1tqUXlZUlmEg$+KSgb2vsvrcvdqBWsEnI+Lv3RJZ+GtbHTe8CyMDgi9E

Verification with correct password: True
Verification with wrong password:   False
--> Password authentication verified.


---
## 4. Session Token Security: Preimage-Resistant Token Hashing
**File:** `backend/app/core/security.py`, `backend/app/models/auth_session.py`  
**Purpose:** Separate cookie tokens from database storage. The raw 256-bit token is sent only in the browser's `HttpOnly` cookie; the database stores only its SHA-256 hash. A database leak does not allow attackers to forge session cookies.


In [5]:
from app.core.security import generate_session_token, hash_session_token

# User logs in -> generate session
raw_cookie_token, db_token_hash = generate_session_token()

print(f"[1] Sent to User Browser (HttpOnly Cookie):")
print(f"    rpm_session = {raw_cookie_token}")

print(f"\n[2] Stored in Database (auth_sessions.token_hash):")
print(f"    token_hash  = {db_token_hash}")

# When user makes a request, browser sends raw cookie:
computed_hash = hash_session_token(raw_cookie_token)
print(f"\n[3] Incoming Cookie SHA-256: {computed_hash}")
print(f"    Matches DB record?        {computed_hash == db_token_hash}")
assert computed_hash == db_token_hash

print("\nSecurity Guarantee:")
print("Because SHA-256 is preimage-resistant, an attacker who steals the database")
print("CANNOT reverse the hash back into the cookie token to hijack active sessions!")


[1] Sent to User Browser (HttpOnly Cookie):
    rpm_session = qkUb5reAQMMqhtOlFiwPhery0POlL_V4mNAbT-EXqaU

[2] Stored in Database (auth_sessions.token_hash):
    token_hash  = bf698cafcfabf50b4b5af5166b47b5909a1a1dfa2150dd967296094e15491759

[3] Incoming Cookie SHA-256: bf698cafcfabf50b4b5af5166b47b5909a1a1dfa2150dd967296094e15491759
    Matches DB record?        True

Security Guarantee:
Because SHA-256 is preimage-resistant, an attacker who steals the database
CANNOT reverse the hash back into the cookie token to hijack active sessions!


---
## 5. Auditability: Tamper-Evident SHA-256 Hash Chaining
**File:** `backend/app/audit/service.py`, `backend/app/models/audit.py`  
**Purpose:** Use Merkle/blockchain-style cryptographic hash chaining. Each audit event embeds the hash of the preceding event:
$$\\text{entry\\_hash} = \\text{SHA-256}(\\text{prev\\_hash} \\parallel \\text{seq} \\parallel \\text{event\\_type} \\parallel \\text{actor} \\parallel \\text{metadata} \\parallel \\text{timestamp})$$
Any retroactive edit, deletion, or insertion breaks the chain immediately.


In [6]:
from sqlalchemy import create_engine
from sqlalchemy.orm import Session
from app.db.base import Base
from app.models import AuditLog
from app.audit.service import write_audit_event, verify_chain_integrity

# Create an in-memory SQLite database for testing
test_engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(test_engine)

with Session(test_engine) as session:
    test_user_id = uuid.uuid4()
    
    # 1. Log three sequential events
    e1 = write_audit_event(session, "USER_LOGIN", actor_id=test_user_id)
    e2 = write_audit_event(session, "CONSENT_GRANTED", actor_id=test_user_id, target_reference="provider-123")
    e3 = write_audit_event(session, "MEASUREMENT_INGESTED", actor_id=test_user_id, target_reference="meas-456")
    session.commit()
    
    print("=== Cryptographic Audit Chain Entries ===")
    entries = session.query(AuditLog).order_by(AuditLog.seq.asc()).all()
    for e in entries:
        print(f"Seq: {e.seq} | Event: {e.event_type:<20} | Prev Hash: {e.prev_hash[:16]}... | Entry Hash: {e.entry_hash[:16]}...")
        
    # Verify chain integrity
    intact = verify_chain_integrity(session)
    print(f"\nInitial Chain Verification: {intact} (Tamper-Free)")
    assert intact is True
    
    # 2. Simulate Attacker Modifying Entry 2 (e.g. rogue DB admin tampering with consent record)
    print("\n--- Simulating Rogue Database Modification ---")
    e2.event_type = "UNAUTHORIZED_ALTERATION"
    session.commit()
    
    # Verify chain again
    tampered = verify_chain_integrity(session)
    print(f"Post-Tampering Chain Verification: {tampered} (TAMPER DETECTED!)")
    assert tampered is False
    print("--> Tamper-evident verification detected the modification successfully!")


=== Cryptographic Audit Chain Entries ===
Seq: 0 | Event: USER_LOGIN           | Prev Hash: 0000000000000000... | Entry Hash: cf2538746178766f...
Seq: 1 | Event: CONSENT_GRANTED      | Prev Hash: cf2538746178766f... | Entry Hash: 2fb692710728d220...
Seq: 2 | Event: MEASUREMENT_INGESTED | Prev Hash: 2fb692710728d220... | Entry Hash: d3f2412646d421b4...

Initial Chain Verification: True (Tamper-Free)

--- Simulating Rogue Database Modification ---
Post-Tampering Chain Verification: False (TAMPER DETECTED!)
--> Tamper-evident verification detected the modification successfully!


---
## 6. Quantum Job Auditing: Circuit Structural Fingerprinting
**File:** `backend/app/ai/quantum/circuit.py`, `backend/app/models/audit.py`  
**Purpose:** Compute a deterministic structural fingerprint of the variational quantum circuit architecture to audit which exact circuit variant ran, without leaking patient vitals into logs.


In [7]:
from app.ai.quantum.circuit import build_autoencoder_circuit, circuit_hash

# Build the 6-qubit autoencoder circuit
circuit, feature_params, weight_params, observables = build_autoencoder_circuit(num_qubits=6, reps=2)

# Compute deterministic structural fingerprint
fingerprint = circuit_hash(circuit)

print("=== Quantum Circuit Provenance ===")
print(f"Qubit Count:      {circuit.num_qubits}")
print(f"Circuit Depth:    {circuit.depth()}")
print(f"Parameter Count:  {len(weight_params)}")
print(f"Structural SHA-256 Fingerprint: {fingerprint}")

print("\nSample Audit Event Logged:")
print(json.dumps({
    "event_type": "QUANTUM_JOB_COMPLETED",
    "circuit_hash": fingerprint,
    "qubit_count": circuit.num_qubits,
    "shots": 1024,
    "note": "Mathematical proof of circuit execution without storing sensitive patient vitals"
}, indent=2))


=== Quantum Circuit Provenance ===
Qubit Count:      6
Circuit Depth:    2
Parameter Count:  18
Structural SHA-256 Fingerprint: 5384420fb55e198a

Sample Audit Event Logged:
{
  "event_type": "QUANTUM_JOB_COMPLETED",
  "circuit_hash": "5384420fb55e198a",
  "qubit_count": 6,
  "shots": 1024,
  "note": "Mathematical proof of circuit execution without storing sensitive patient vitals"
}


---
## Summary: Cryptography Matrix
| Domain | Primitive | How Data Looks / Is Transmitted | Code Source |
|---|---|---|---|
| **Data at Rest** | AES-256-GCM + Nonce | Ingests `78.5` $\\to$ Stored as `\\x8e\\x12...` binary blob | `app.core.crypto` |
| **Data in Transit** | Data Minimization + TLS | Push packet carries `{"alert_id": "..."}`, never raw vitals | `app.alerts.engine` |
| **Authentication** | Argon2id | Password $\\to$ `$argon2id$v=19$m=65536,t=3,p=4$...` | `app.core.security` |
| **Session Control** | CSPRNG + SHA-256 | Cookie: `raw_token` $\\to$ DB: `sha256(raw_token)` | `app.core.security` |
| **Audit Log** | SHA-256 Hash Chain | Each row hashes `prev_hash`; edits break validation | `app.audit.service` |
| **Quantum Audit** | Circuit SHA-256 Hash | Hashes circuit topology; no vitals logged | `app.ai.quantum.circuit` |
